In [57]:
from google.colab import drive

drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [58]:
folder_path = '/content/drive/MyDrive/Startup Data Analysis/Data/raw'

In [59]:
import pandas as pd
import numpy as np
import os

In [60]:
os.listdir(folder_path)

['Indian_startup_data.csv']

In [61]:
data = pd.read_csv(f'{folder_path}/Indian_startup_data.csv')

In [62]:
print("Shape:", data.shape)
data.info()

print("\nMissing values per column:")
print(data.isnull().sum())

print("\nFull-row duplicates:", data.duplicated().sum())

Shape: (4317, 8)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4317 entries, 0 to 4316
Data columns (total 8 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   date             4315 non-null   object 
 1   startup          4317 non-null   object 
 2   industry         4317 non-null   object 
 3   sub_vertical     4145 non-null   object 
 4   city             4122 non-null   object 
 5   investors        4194 non-null   object 
 6   investment_type  4303 non-null   object 
 7   amount_usd       3322 non-null   float64
dtypes: float64(1), object(7)
memory usage: 269.9+ KB

Missing values per column:
date                 2
startup              0
industry             0
sub_vertical       172
city               195
investors          123
investment_type     14
amount_usd         995
dtype: int64

Full-row duplicates: 0


In [63]:
# 1. date, investors - dropping these rows
data_clean = data.dropna(subset=['date', 'investors']).copy()
print(f"Rows before: {len(data)} -> after dropping missing date/investors: {len(data_clean)}")

# 2. investment_type — only 14 missing - filling unknown
data_clean['investment_type'] = data_clean['investment_type'].fillna('Unknown')

# 3. sub_vertical, city - filling unkknown
data_clean['sub_vertical'] = data_clean['sub_vertical'].fillna('Unknown')
data_clean['city'] = data_clean['city'].fillna('Unknown')

# 4. amount_usd — not dropping
data_clean['amount_disclosed'] = data_clean['amount_usd'].notnull()
print(f"\nDeals with disclosed amount: {data_clean['amount_disclosed'].sum()}")
print(f"Deals with undisclosed amount: {(~data_clean['amount_disclosed']).sum()}")

print(f"\nFinal shape: {data_clean.shape}")

Rows before: 4317 -> after dropping missing date/investors: 4192

Deals with disclosed amount: 3219
Deals with undisclosed amount: 973

Final shape: (4192, 9)


In [64]:
dup_count = data_clean.duplicated().sum()
print("Exact duplicates:", dup_count)

data_clean = data_clean.drop_duplicates().reset_index(drop=True)
print("Shape after dropping exact duplicates:", data_clean.shape)

Exact duplicates: 0
Shape after dropping exact duplicates: (4192, 9)


In [65]:
soft_dupes = data_clean.duplicated(subset=['startup', 'date'], keep=False)
print("Possible soft duplicates:", soft_dupes.sum())

data_clean[soft_dupes].sort_values(['startup', 'date'])

Possible soft duplicates: 14


,date,startup,industry,sub_vertical,city,investors,investment_type,amount_usd,amount_disclosed
3006,18/03/2020,DriveEdge,Food & Beverage,Food Delivery,Kolkata,"Lightspeed India, Y Combinator",Pre-Series A,196000.0,True
3007,18/03/2020,DriveEdge,Logistics,Last Mile,Pune,"Omidyar Network, Tiger Global, Y Combinator",Seed/Angel,200000.0,True
4021,29/07/2024,HomeAnalytics,E-commerce,Grocery,Pune,"Ribbit Capital, Zodius Capital",Seed/Angel,517000.0,True
4022,29/07/2024,HomeAnalytics,FinTech,NeoBank,Kolkata,Zodius Capital,Seed/Angel,118000.0,True
3735,19/04/2023,Housejoy,EdTech,K12,Mumbai,Lightspeed India,Seed/Angel,199000.0,True
3736,19/04/2023,Housejoy,E-commerce,E-Retail,Gurugram,"Kedaara Capital, Ribbit Capital, Y Combinator",Series C,142883000.0,True
3821,17/09/2023,HyperLoop,AgriTech,Supply Chain,Mumbai,"Info Edge (India), Mirae Asset",Seed/Angel,113000.0,True
3822,17/09/2023,HyperLoop,Real Estate,Rental Tech,Bengaluru,Prosus Ventures,Seed/Angel,62000.0,True
3208,27/10/2020,Origo,AgriTech,Agricultural Commodities Management,Gurugram,Northern Arc Capital,Debt,4700000.0,True
3209,27/10/2020,Origo,Media & Entertainment,Online Video Editing,Mumbai,Sequoia Capital,Series A,15000000.0,True


In [66]:
data_clean['date'] = pd.to_datetime(data_clean['date'], format='%d/%m/%Y', errors='coerce')
print("Unparseable dates:", data_clean['date'].isnull().sum())
data_clean['year'] = data_clean['date'].dt.year

Unparseable dates: 0


In [67]:
data_clean['investor_count'] = data_clean['investors'].apply(
    lambda x: len([i.strip() for i in str(x).split(',') if i.strip() != ''])
)

data_clean[['investors', 'investor_count']].head(10)

,investors,investor_count
0,"IDG Ventures, TPG Growth, TR Capital",3
1,"Srinivasa Rao Paturi, Sudhakar Reddy, Venkat V...",3
2,TO THE NEW Ventures,1
3,"Haldyn Glass, Rohan Ajila, Tom Clausen",3
4,"DeNA Co, Japan, Kris GopalKrishnan), Teruhide ...",4
5,Matrix Partners,1
6,"Akash Agarwal, Anupam Mittal, Gaurav Agarwal, ...",7
7,SRI Capital,1
8,IvyCap Ventures,1
9,"Blume Ventures, Jungle Ventures, Redbright Par...",3


In [68]:
data_clean.insert(0, 'deal_id', range(1, len(data_clean) + 1))
data_clean.head()

,deal_id,date,startup,industry,sub_vertical,city,investors,investment_type,amount_usd,amount_disclosed,year,investor_count
0,1,2015-01-02,Lenskart,E-commerce,Unknown,Unknown,"IDG Ventures, TPG Growth, TR Capital",Private Equity,2150000.0,True,2015,3
1,2,2015-01-02,VioletStreet,Unknown,Unknown,Unknown,"Srinivasa Rao Paturi, Sudhakar Reddy, Venkat V...",Seed/Angel,315000.0,True,2015,3
2,3,2015-01-05,#Fame,Unknown,Unknown,Unknown,TO THE NEW Ventures,Private Equity,10000000.0,True,2015,1
3,4,2015-01-05,Gympik,Healthcare,Unknown,Unknown,"Haldyn Glass, Rohan Ajila, Tom Clausen",Seed/Angel,135000.0,True,2015,3
4,5,2015-01-05,Lookup,Media & Entertainment,Unknown,Unknown,"DeNA Co, Japan, Kris GopalKrishnan), Teruhide ...",Seed/Angel,380000.0,True,2015,4


In [69]:
data_clean['date'] = data_clean['date'].dt.strftime('%d/%m/%Y')
data_clean.head()

,deal_id,date,startup,industry,sub_vertical,city,investors,investment_type,amount_usd,amount_disclosed,year,investor_count
0,1,02/01/2015,Lenskart,E-commerce,Unknown,Unknown,"IDG Ventures, TPG Growth, TR Capital",Private Equity,2150000.0,True,2015,3
1,2,02/01/2015,VioletStreet,Unknown,Unknown,Unknown,"Srinivasa Rao Paturi, Sudhakar Reddy, Venkat V...",Seed/Angel,315000.0,True,2015,3
2,3,05/01/2015,#Fame,Unknown,Unknown,Unknown,TO THE NEW Ventures,Private Equity,10000000.0,True,2015,1
3,4,05/01/2015,Gympik,Healthcare,Unknown,Unknown,"Haldyn Glass, Rohan Ajila, Tom Clausen",Seed/Angel,135000.0,True,2015,3
4,5,05/01/2015,Lookup,Media & Entertainment,Unknown,Unknown,"DeNA Co, Japan, Kris GopalKrishnan), Teruhide ...",Seed/Angel,380000.0,True,2015,4


In [70]:
investor_rows = []
for _, row in data_clean.iterrows():
    investors_list = [i.strip() for i in str(row['investors']).split(',') if i.strip() != '']
    for inv in investors_list:
        investor_rows.append({'deal_id': row['deal_id'], 'investor': inv})

investor_df = pd.DataFrame(investor_rows)
print("Total investor-deal rows:", len(investor_df))
investor_df.head(10)

Total investor-deal rows: 8057


,deal_id,investor
0,1,IDG Ventures
1,1,TPG Growth
2,1,TR Capital
3,2,Srinivasa Rao Paturi
4,2,Sudhakar Reddy
5,2,Venkat Vallabhneni
6,3,TO THE NEW Ventures
7,4,Haldyn Glass
8,4,Rohan Ajila
9,4,Tom Clausen


In [71]:
output_folder = '/content/drive/MyDrive/Startup Data Analysis/Data/cleaned'
os.makedirs(output_folder, exist_ok=True)

data_clean.to_csv(f'{output_folder}/Indian_startup_data_cleaned.csv', index=False)
investor_df.to_csv(f'{output_folder}/deal_investors.csv', index=False)

print("Saved files:", os.listdir(output_folder))

Saved files: ['Indian_startup_data_cleaned.csv', 'deal_investors.csv']
